# Sales Data Analytics Dashboard using Python
**Author:** Archit Mahajan  
**Program:** IBM SkillsBuild Data Analytics with AI Internship 2026 (BharatCares | AICTE)

This notebook loads retail sales data, cleans it, explores trends, and forecasts future monthly sales with Linear Regression. The same logic powers the Streamlit dashboard (`app.py`).

## 1. Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True
os.makedirs("charts", exist_ok=True)

## 2. Load the dataset
Primary dataset: **Superstore Sales Dataset** (Kaggle): https://www.kaggle.com/datasets/vivek468/superstore-dataset-final  
Download `Sample - Superstore.csv` and place it next to this notebook. If the file is not found, the notebook generates a small sample dataset with the same columns so that the code still runs end to end.

In [ ]:
def make_sample_data(seed=42, n=8000):
    """Sample retail data with the same columns used from Superstore."""
    rng = np.random.default_rng(seed)
    dates = pd.to_datetime("2022-01-01") + pd.to_timedelta(rng.integers(0, 4 * 365, n), unit="D")
    category = rng.choice(["Furniture", "Office Supplies", "Technology"], n, p=[0.25, 0.55, 0.20])
    base = {"Furniture": 350, "Office Supplies": 90, "Technology": 450}
    margin = {"Furniture": 0.04, "Office Supplies": 0.17, "Technology": 0.16}
    growth = 1 + 0.25 * (dates.year - 2022) + 0.15 * np.isin(dates.month, [11, 12])
    sales = np.array([base[c] for c in category]) * rng.lognormal(0, 0.35, n) * growth
    profit = sales * np.array([margin[c] for c in category]) + rng.normal(0, 8, n)
    return pd.DataFrame({
        "Order Date": dates,
        "Region": rng.choice(["West", "East", "Central", "South"], n, p=[0.32, 0.29, 0.23, 0.16]),
        "Category": category,
        "Sub-Category": rng.choice(["Chairs", "Tables", "Phones", "Binders", "Paper", "Accessories", "Storage"], n),
        "Segment": rng.choice(["Consumer", "Corporate", "Home Office"], n, p=[0.52, 0.30, 0.18]),
        "Sales": sales.round(2),
        "Quantity": rng.integers(1, 10, n),
        "Profit": profit.round(2),
    })

CSV_PATH = "Sample - Superstore.csv"
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, encoding="latin-1")
    DATA_SOURCE = "Kaggle Superstore dataset"
else:
    df = make_sample_data()
    DATA_SOURCE = "Generated sample data (Superstore-style columns)"
print("Data source:", DATA_SOURCE)
print("Shape:", df.shape)
df.head()

## 3. Data cleaning

In [ ]:
df = df.drop_duplicates()
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df = df.dropna(subset=["Order Date", "Sales"])
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.to_period("M")
print("Missing values per column:\n", df.isna().sum())
print("Rows after cleaning:", len(df))

## 4. Key performance indicators

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
orders = len(df)
margin_pct = total_profit / total_sales * 100
print(f"Total Sales   : {total_sales:,.2f}")
print(f"Total Profit  : {total_profit:,.2f}")
print(f"Profit Margin : {margin_pct:.2f}%")
print(f"Records       : {orders:,}")

## 5. Exploratory data analysis

In [ ]:
monthly = df.groupby("Month")["Sales"].sum().reset_index()
monthly["Month"] = monthly["Month"].dt.to_timestamp()
plt.plot(monthly["Month"], monthly["Sales"], marker="o", color="#1f77b4")
plt.title("Monthly Sales Trend"); plt.xlabel("Month"); plt.ylabel("Sales")
plt.tight_layout(); plt.savefig("charts/monthly_sales.png", dpi=150); plt.show()

In [ ]:
cat = df.groupby("Category")[["Sales", "Profit"]].sum().sort_values("Sales", ascending=False)
cat.plot(kind="bar", color=["#1f77b4", "#2ca02c"])
plt.title("Sales and Profit by Category"); plt.ylabel("Amount"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig("charts/category.png", dpi=150); plt.show()

In [ ]:
reg = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
plt.pie(reg, labels=reg.index, autopct="%1.1f%%", startangle=90)
plt.title("Sales Share by Region"); plt.grid(False)
plt.tight_layout(); plt.savefig("charts/region.png", dpi=150); plt.show()

In [ ]:
top = df.groupby("Sub-Category")["Sales"].sum().sort_values().tail(7)
top.plot(kind="barh", color="#ff7f0e")
plt.title("Top Sub-Categories by Sales"); plt.xlabel("Sales")
plt.tight_layout(); plt.savefig("charts/subcategory.png", dpi=150); plt.show()

## 6. Sales forecasting with Linear Regression
Monthly sales are converted to a time index (0, 1, 2, ...). Two Linear Regression models are compared:
1. **Trend only**: sales vs. time index (a straight line).
2. **Trend + Seasonality**: the time index plus one-hot month-of-year columns, so the model can learn that some months (for example November and December) are always higher.

The last 12 months are held out as a test set.

In [ ]:
monthly = monthly.reset_index(drop=True)
monthly["t"] = np.arange(len(monthly))
monthly["month_no"] = monthly["Month"].dt.month
y = monthly["Sales"]

def build_features(frame, seasonal):
    X = frame[["t"]].copy()
    if seasonal:
        for m in range(2, 13):          # month 1 is the baseline
            X[f"m{m}"] = (frame["month_no"] == m).astype(int)
    return X

TEST_MONTHS = 12
split = len(monthly) - TEST_MONTHS
results = {}
for name, seasonal in [("Trend only", False), ("Trend + Seasonality", True)]:
    X = build_features(monthly, seasonal)
    model = LinearRegression().fit(X.iloc[:split], y.iloc[:split])
    pred = model.predict(X.iloc[split:])
    results[name] = {
        "MAE": mean_absolute_error(y.iloc[split:], pred),
        "RMSE": np.sqrt(mean_squared_error(y.iloc[split:], pred)),
        "R2": r2_score(y.iloc[split:], pred),
    }
comparison = pd.DataFrame(results).T
print(comparison.round(3))

The seasonal model gives lower errors and a higher R2, so it is used for the final forecast.

In [ ]:
X_all = build_features(monthly, True)
final_model = LinearRegression().fit(X_all, y)

future_dates = pd.date_range(monthly["Month"].iloc[-1] + pd.offsets.MonthBegin(1), periods=6, freq="MS")
future = pd.DataFrame({"t": np.arange(len(monthly), len(monthly) + 6), "month_no": future_dates.month})
forecast = pd.DataFrame({"Month": future_dates,
                         "Forecast Sales": final_model.predict(build_features(future, True))})
print(forecast.round({"Forecast Sales": 2}))

plt.plot(monthly["Month"], y, label="Actual", color="#1f77b4")
plt.plot(monthly["Month"], final_model.predict(X_all), "--", label="Model fit", color="gray")
plt.plot(forecast["Month"], forecast["Forecast Sales"], "o-", label="Forecast (next 6 months)", color="#d62728")
plt.title("Sales Forecast using Linear Regression (Trend + Seasonality)")
plt.xlabel("Month"); plt.ylabel("Sales"); plt.legend()
plt.tight_layout(); plt.savefig("charts/forecast.png", dpi=150); plt.show()

## 7. Insights (computed from the data)

In [ ]:
best_cat = cat["Sales"].idxmax()
margins = (cat["Profit"] / cat["Sales"] * 100).sort_values()
best_region = reg.idxmax()
peak_month = df.groupby(df["Order Date"].dt.month)["Sales"].sum().idxmax()
print(f"Top category by sales : {best_cat}")
print(f"Lowest profit margin  : {margins.index[0]} ({margins.iloc[0]:.1f}%)")
print(f"Highest profit margin : {margins.index[-1]} ({margins.iloc[-1]:.1f}%)")
print(f"Top region by sales   : {best_region}")
print(f"Peak sales month      : {peak_month}")
avg_by_month = monthly.groupby(monthly["Month"].dt.month)["Sales"].mean()
print(f"Strongest month (avg)  : {avg_by_month.idxmax()}   Weakest month (avg): {avg_by_month.idxmin()}")

## 8. Streamlit dashboard
The interactive dashboard is in `app.py`. Run it with: `streamlit run app.py`